# CW1: Cloud AI Framework for Credit Card Fraud Detection
## Notebook 3: Deployment and Monitoring on Azure Machine Learning

This notebook covers the deployment and monitoring stage of the AI lifecycle. It uses the Azure Machine Learning Python SDK (v2) to move the model built in Notebook 2 into the cloud:

1. Connect securely to the Azure Machine Learning workspace
2. Upload the dataset as a versioned data asset
3. Register the trained model with its metadata
4. Define the scoring script and runtime environment
5. Deploy the model to a managed online endpoint (real-time serving)
6. Test the endpoint and measure response time
7. Review logs and monitoring, then delete the endpoint to control cost

---

### 1. Connecting to the Azure Machine Learning Workspace

This step authenticates with Microsoft Entra ID (Azure Active Directory) and connects to the `mlw-fraud-cw1` workspace using the `config.json` file downloaded from the Azure portal. No passwords or keys are stored in the notebook: authentication happens through a browser sign-in, following the principle of keeping credentials out of code.


In [2]:
import os
import json
import time
import azure.ai.ml
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

# Authenticate: use an existing Azure sign-in if available, otherwise open a browser
try:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    credential.get_token("https://management.azure.com/.default")
    print("Authenticated with existing Azure credentials.")
except Exception:
    credential = InteractiveBrowserCredential()
    print("Opening a browser window to sign in with your Bradford account...")

# Connect to the workspace using config.json
ml_client = MLClient.from_config(credential=credential, path="config.json")
workspace = ml_client.workspaces.get(ml_client.workspace_name)

print(f"\nazure-ai-ml version: {azure.ai.ml.__version__}")
print(f"Workspace:      {workspace.name}")
print(f"Resource group: {ml_client.resource_group_name}")
print(f"Location:       {workspace.location}")
print(f"Subscription:   {ml_client.subscription_id[:8]}... (truncated)")

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' pa

Opening a browser window to sign in with your Bradford account...

azure-ai-ml version: 1.35.2
Workspace:      mlw-fraud-cw1
Resource group: rg-fraud-cw1
Location:       swedencentral
Subscription:   88f04469... (truncated)


Invalid type `<class 'type'>` for attribute value. Expected one of bool, str, None, bytes, int, float or a Mapping or Sequence of those types. Value's __str__ method will be called if it exists, otherwise the value will be replaced with None.


### 2. Data Ingestion: Registering the Dataset as a Data Asset

This step uploads the raw transaction file to the workspace's Azure Blob Storage account and registers it as a **versioned data asset**. Versioning records exactly which data each model was trained on (data lineage), which supports reproducibility and auditing. The data is encrypted at rest by default with Microsoft-managed keys.


In [3]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.core.exceptions import ResourceNotFoundError

DATA_NAME = "creditcard-fraud-raw"
DATA_VERSION = "1"

try:
    data_asset = ml_client.data.get(name=DATA_NAME, version=DATA_VERSION)
    print(f"Data asset already registered, reusing it: {DATA_NAME} v{DATA_VERSION}")
except ResourceNotFoundError:
    print(f"Uploading {os.path.getsize('data/creditcard.csv') / 1024**2:.0f} MB to Azure Blob Storage...")
    start = time.time()
    data_asset = ml_client.data.create_or_update(Data(
        name=DATA_NAME,
        version=DATA_VERSION,
        type=AssetTypes.URI_FILE,
        path=os.path.join("data", "creditcard.csv"),
        description="ULB credit card fraud dataset (Kaggle mlg-ulb/creditcardfraud): "
                    "284,807 transactions, 492 fraud, PCA-anonymised features.",
        tags={"source": "kaggle", "rows": "284807", "fraud_rate": "0.173%",
              "project": "COS7054-CW1"},
    ))
    print(f"Upload finished in {time.time() - start:.0f} s.")

print(f"\nName:     {data_asset.name}")
print(f"Version:  {data_asset.version}")
print(f"Type:     {data_asset.type}")
print(f"Location: {data_asset.path}")

Uploading 144 MB to Azure Blob Storage...


Uploading creditcard.csv (< 1 MB): 4.19MB [01:53, 36.9kB/s]                                                            




Upload finished in 123 s.

Name:     creditcard-fraud-raw
Version:  1
Type:     uri_file
Location: azureml://subscriptions/88f04469-ed80-43d5-877e-2d9e62dd2d6e/resourcegroups/rg-fraud-cw1/workspaces/mlw-fraud-cw1/datastores/workspaceblobstore/paths/LocalUpload/715bf0709a532bd92bf394cd57dc8c4cbf503faf27142ae8ef10119829a672a8/creditcard.csv


### 3. Model Registration

This step registers the trained model in the workspace's **model registry**. The registry stores each model version with its files, metrics and lineage, so the team can see which model is in production, what data it was trained on, and how it performed. This supports auditing and rollback, which regulators expect for automated decisions in financial services.

The registered package contains three files:
- `fraud_xgb_model.json`: the trained XGBoost model
- `amount_scaler.joblib`: the fitted scaler for the `Amount` feature
- `model_metadata.json`: feature order, decision threshold, hyperparameters and test metrics

The model's test metrics are attached as tags, so they are visible directly in Azure Machine Learning studio.

In [4]:
import shutil
from azure.ai.ml.entities import Model

# Package only the files the endpoint needs
PACKAGE_DIR = os.path.join("deploy", "model")
os.makedirs(PACKAGE_DIR, exist_ok=True)
for f in ["fraud_xgb_model.json", "amount_scaler.joblib", "model_metadata.json"]:
    shutil.copy(os.path.join("models", f), PACKAGE_DIR)
print("Model package contents:", sorted(os.listdir(PACKAGE_DIR)))

with open(os.path.join(PACKAGE_DIR, "model_metadata.json")) as f:
    meta = json.load(f)
metrics = meta["test_metrics"]

MODEL_NAME = "fraud-xgb-sqrt-weight"
registered_model = ml_client.models.create_or_update(Model(
    name=MODEL_NAME,
    path=PACKAGE_DIR,
    type=AssetTypes.CUSTOM_MODEL,
    description="XGBoost credit card fraud classifier with square-root class weighting. "
                f"Decision threshold {meta['decision_threshold']}.",
    tags={
        "algorithm": "XGBoost",
        "threshold": str(meta["decision_threshold"]),
        "test_pr_auc": str(metrics["pr_auc"]),
        "test_roc_auc": str(metrics["roc_auc"]),
        "test_precision": str(metrics["precision"]),
        "test_recall": str(metrics["recall"]),
        "training_data": f"azureml:{DATA_NAME}:{DATA_VERSION}",
        "project": "COS7054-CW1",
    },
))

print(f"\nRegistered model: {registered_model.name}")
print(f"Version:          {registered_model.version}")
print(f"Threshold:        {meta['decision_threshold']}")
print(f"Test PR-AUC:      {metrics['pr_auc']}")
print(f"Test recall:      {metrics['recall']}")
print(f"Linked data:      azureml:{DATA_NAME}:{DATA_VERSION}")

Model package contents: ['amount_scaler.joblib', 'fraud_xgb_model.json', 'model_metadata.json']


Uploading model (0.57 MBs): 100%|###########################################| 571702/571702 [00:07<00:00, 80184.50it/s]





Registered model: fraud-xgb-sqrt-weight
Version:          1
Threshold:        0.3861
Test PR-AUC:      0.8189
Test recall:      0.7789
Linked data:      azureml:creditcard-fraud-raw:1


### 4. Scoring Script

The scoring script is the code that runs inside the cloud endpoint. Azure Machine Learning calls two functions:

- **`init()`** runs once when the container starts. It loads the model, the scaler and the metadata into memory, so individual requests stay fast.
- **`run()`** runs for every request. It validates the input, applies **exactly the same feature engineering as in training** (log-transformed and scaled `Amount`, cyclical hour encoding), scores each transaction, and applies the tuned decision threshold.

The script also includes basic defensive measures: it rejects requests with missing fields or missing values instead of producing a silent wrong prediction, and it logs only operational information (transaction counts, number flagged, latency). It never logs raw transaction data, in line with the GDPR principle of data minimisation.

In [7]:
%%writefile deploy/src/score.py
"""Scoring script for the credit card fraud detection endpoint.

Azure Machine Learning calls init() once when the container starts and
run() for every scoring request.
"""
import json
import logging
import os
import time

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

logger = logging.getLogger("fraud-endpoint")
logger.setLevel(logging.INFO)

model = scaler = metadata = None


def _find(filename):
    """Locate a file inside the registered model folder."""
    for root, _, files in os.walk(os.environ["AZUREML_MODEL_DIR"]):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(filename)


def init():
    global model, scaler, metadata
    model = xgb.XGBClassifier()
    model.load_model(_find("fraud_xgb_model.json"))
    scaler = joblib.load(_find("amount_scaler.joblib"))
    with open(_find("model_metadata.json")) as f:
        metadata = json.load(f)
    logger.info("Model loaded: %s, threshold %.4f",
                metadata["model_name"], metadata["decision_threshold"])


def _preprocess(df):
    """Apply exactly the same feature engineering as in training."""
    hour = (df["Time"] // 3600) % 24
    out = df.copy()
    out["log_Amount"] = scaler.transform(
        pd.DataFrame({"log_Amount": np.log1p(df["Amount"])}))[:, 0]
    out["Hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["Hour_cos"] = np.cos(2 * np.pi * hour / 24)
    return out[metadata["model_features"]]


def run(raw_data):
    start = time.perf_counter()
    try:
        records = json.loads(raw_data)["data"]
        df = pd.DataFrame(records)
        missing = [c for c in metadata["raw_input_columns"] if c not in df.columns]
        if missing:
            return {"error": f"Missing required fields: {missing}"}
        if df[metadata["raw_input_columns"]].isna().any().any():
            return {"error": "Input contains missing values."}

        proba = model.predict_proba(_preprocess(df))[:, 1]
        threshold = metadata["decision_threshold"]
        results = [{"fraud_probability": round(float(p), 6),
                    "is_fraud": bool(p >= threshold)} for p in proba]

        # Operational log only: counts and latency, never transaction data
        latency_ms = (time.perf_counter() - start) * 1000
        logger.info("Scored %d transactions, %d flagged, %.2f ms",
                    len(results), sum(r["is_fraud"] for r in results), latency_ms)
        return {"model": metadata["model_name"], "threshold": threshold,
                "predictions": results}
    except Exception as exc:
        logger.error("Scoring failed: %s", exc)
        return {"error": str(exc)}

Writing deploy/src/score.py


### 5. Runtime Environment and Local Pre-Deployment Test

This step defines the software environment the endpoint container will run: a Microsoft-maintained Ubuntu base image plus a Conda specification listing the Python packages. Package versions are **pinned to those used in training**, because a different version of XGBoost or scikit-learn could load the model differently and silently change predictions. Pinning versions makes the deployment reproducible.

Before deploying, the scoring script is tested locally, exactly as Azure will call it: `init()` loads the model package and `run()` scores the sample request. The predictions are compared against those saved in Notebook 2. Catching errors here is much faster and cheaper than debugging a failed cloud deployment, which takes 10–15 minutes per attempt.

In [9]:
import importlib.util
import pandas as pd
import xgboost, sklearn, numpy, joblib as joblib_pkg
from azure.ai.ml.entities import Environment

# 1. Conda specification pinned to the training library versions
conda_spec = f"""name: fraud-endpoint-env
channels:
  - conda-forge
dependencies:
  - python=3.11
  - pip
  - pip:
      - azureml-inference-server-http
      - xgboost=={xgboost.__version__}
      - scikit-learn=={sklearn.__version__}
      - pandas=={pd.__version__}
      - numpy=={numpy.__version__}
      - joblib=={joblib_pkg.__version__}
"""
CONDA_PATH = os.path.join("deploy", "conda.yaml")
with open(CONDA_PATH, "w") as f:
    f.write(conda_spec)
print("Environment specification (deploy/conda.yaml):")
print(conda_spec)

env = Environment(
    name="fraud-xgb-env",
    description="Inference environment for the fraud detection endpoint",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    conda_file=CONDA_PATH,
)

# 2. Local test: load score.py and call it the way Azure will
os.environ["AZUREML_MODEL_DIR"] = "deploy"            # contains deploy/model/
spec = importlib.util.spec_from_file_location("score", os.path.join("deploy", "src", "score.py"))
score = importlib.util.module_from_spec(spec)
spec.loader.exec_module(score)

score.init()
with open(os.path.join("models", "sample_request.json")) as f:
    sample_request = f.read()
response = score.run(sample_request)

local = pd.DataFrame(response["predictions"])
expected = pd.read_csv(os.path.join("models", "sample_expected_output.csv"))
comparison = pd.DataFrame({
    "true_class": expected["true_class"],
    "expected_probability": expected["fraud_probability"].round(6),
    "local_probability": local["fraud_probability"],
    "flagged": local["is_fraud"],
})
print("Local scoring test against Notebook 2 predictions:")
display(comparison)

max_diff = (comparison["expected_probability"] - comparison["local_probability"]).abs().max()
assert max_diff < 1e-5, "Local scoring does not match Notebook 2!"
print(f"\nPassed: maximum difference {max_diff:.1e}. score.py reproduces Notebook 2 exactly.")

# 3. Input validation test
bad = score.run(json.dumps({"data": [{"Time": 0, "Amount": 10.0}]}))
print(f"Invalid request correctly rejected: {'error' in bad}")

Environment specification (deploy/conda.yaml):
name: fraud-endpoint-env
channels:
  - conda-forge
dependencies:
  - python=3.11
  - pip
  - pip:
      - azureml-inference-server-http
      - xgboost==3.2.0
      - scikit-learn==1.6.1
      - pandas==2.2.3
      - numpy==2.1.3
      - joblib==1.4.2

Local scoring test against Notebook 2 predictions:


,true_class,expected_probability,local_probability,flagged
0,1,0.001333,0.001333,False
1,1,0.997782,0.997782,True
2,1,0.972561,0.972561,True
3,0,0.000056,0.000056,False
4,0,0.000135,0.000135,False
5,0,0.000043,0.000043,False



Passed: maximum difference 0.0e+00. score.py reproduces Notebook 2 exactly.
Invalid request correctly rejected: True


### 6. Deployment to a Managed Online Endpoint

This step deploys the model as a real-time web service. Azure Machine Learning uses two concepts:

- **Endpoint:** a stable HTTPS address that client applications (for example, a bank's payment authorisation system) call. It is secured with **key-based authentication**: every request must include a secret key, and traffic is encrypted with TLS.
- **Deployment:** the model, scoring script, environment and compute behind the endpoint. An endpoint can host several deployments and split traffic between them (for example, 90% to the current model and 10% to a new one), which enables safe **blue-green** or canary releases without downtime.

This is a **Platform-as-a-Service (PaaS)** capability: Azure provisions the virtual machine, builds the container image, handles scaling, health checks and operating system patching, while we supply only the model and code.

**Compute:** one `Standard_F4s_v2` instance (4 vCPUs, 8 GB RAM), a compute-optimised CPU machine within the subscription's quota. XGBoost needs no GPU. **Application Insights** is enabled to collect request logs and performance metrics for monitoring.

The first deployment builds a container image and usually takes **10–20 minutes**.

In [12]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, CodeConfiguration

# Endpoint names must be unique within the Azure region, so a short suffix is added
ENDPOINT_NAME = f"fraud-endpoint-{int(time.time()) % 100000}"
print(f"Endpoint name: {ENDPOINT_NAME}")

# 1. Create the endpoint (the stable, key-secured HTTPS address)
endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Real-time credit card fraud scoring (COS7054 CW1)",
    auth_mode="key",
    tags={"project": "COS7054-CW1", "model": MODEL_NAME},
)
start = time.time()
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Endpoint created in {time.time() - start:.0f} s.\n")

# 2. Create the deployment (model + code + environment + compute)
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=f"azureml:{registered_model.name}:{registered_model.version}",
    code_configuration=CodeConfiguration(code=os.path.join("deploy", "src"),
                                         scoring_script="score.py"),
    environment=env,
    instance_type="Standard_F4s_v2",
    instance_count=1,
    app_insights_enabled=True,
)
print("Deploying the model: building the container image and provisioning the VM.")
print("This usually takes 10-20 minutes. Please wait...")
start = time.time()
ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"Deployment completed in {(time.time() - start) / 60:.1f} minutes.\n")

# 3. Route 100% of traffic to the new deployment
endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
print(f"Endpoint:          {endpoint.name}")
print(f"Provisioning:      {endpoint.provisioning_state}")
print(f"Authentication:    {endpoint.auth_mode}")
print(f"Traffic:           {endpoint.traffic}")
print(f"Scoring URI:       {endpoint.scoring_uri}")

Endpoint name: fraud-endpoint-82295


Check: endpoint fraud-endpoint-82295 exists


Endpoint created in 104 s.

Deploying the model: building the container image and provisioning the VM.
This usually takes 10-20 minutes. Please wait...


Uploading src (0.01 MBs): 100%|#################################################| 7899/7899 [00:00<00:00, 10803.64it/s]




.........................................................................................................................................................................Deployment completed in 18.7 minutes.

Endpoint:          fraud-endpoint-82295
Provisioning:      EndpointProvisioningState.SUCCEEDED
Authentication:    key
Traffic:           {'blue': 100}
Scoring URI:       https://fraud-endpoint-82295.swedencentral.inference.ml.azure.com/score


### 7. Testing the Live Endpoint

This step verifies the deployed service from the client's point of view, as a bank's payment system would call it:

1. **Functional test:** the six sample transactions are sent through the SDK, and the cloud predictions are compared with the local predictions.
2. **Consistency test:** 1,000 raw test-set transactions are sent over HTTPS in batches of 100, to confirm that the cloud endpoint reproduces Notebook 2 at scale.
3. **Latency test:** 30 single-transaction requests measure the end-to-end response time a payment system would experience, including the network round-trip.
4. **Security test:** a request without the authentication key must be refused.

The endpoint key is retrieved securely through the SDK and is never printed or saved.

In [13]:
import requests
import numpy as np
from sklearn.model_selection import train_test_split

# Secure retrieval of the endpoint key (never printed)
keys = ml_client.online_endpoints.get_keys(name=ENDPOINT_NAME)
headers = {"Content-Type": "application/json", "Authorization": f"Bearer {keys.primary_key}"}
scoring_uri = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri

# 1. Functional test with the sample request through the SDK
cloud_response = json.loads(ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME, deployment_name="blue",
    request_file=os.path.join("models", "sample_request.json")))
cloud = pd.DataFrame(cloud_response["predictions"])
functional = comparison[["true_class", "expected_probability"]].copy()
functional["cloud_probability"] = cloud["fraud_probability"]
functional["cloud_flagged"] = cloud["is_fraud"]
print("1. Functional test: sample request scored by the cloud endpoint")
display(functional)

# 2. Consistency test: 1,000 raw test transactions over HTTPS
raw = pd.read_csv(os.path.join("data", "creditcard.csv")).drop_duplicates(keep="first").reset_index(drop=True)
_, raw_test, _, raw_y_test = train_test_split(raw.drop(columns="Class"), raw["Class"],
                                              test_size=0.20, stratify=raw["Class"], random_state=42)
batch = raw_test.iloc[:1000]
cloud_proba = []
for i in range(0, len(batch), 100):
    payload = json.dumps({"data": batch.iloc[i:i + 100].to_dict(orient="records")})
    r = requests.post(scoring_uri, data=payload, headers=headers, timeout=30)
    r.raise_for_status()
    cloud_proba += [p["fraud_probability"] for p in r.json()["predictions"]]
local_proba = [p["fraud_probability"] for p in
               score.run(json.dumps({"data": batch.to_dict(orient="records")}))["predictions"]]
consistency_diff = np.abs(np.array(cloud_proba) - np.array(local_proba)).max()
print(f"\n2. Consistency test: {len(cloud_proba)} transactions scored in the cloud "
      f"({int(raw_y_test.iloc[:1000].sum())} frauds)")
print(f"   Maximum difference between cloud and local predictions: {consistency_diff:.1e}")

# 3. Latency test: 30 single-transaction requests
single = json.dumps({"data": [raw_test.iloc[0].to_dict()]})
requests.post(scoring_uri, data=single, headers=headers, timeout=30)   # warm-up call
latencies = []
for _ in range(30):
    t0 = time.perf_counter()
    requests.post(scoring_uri, data=single, headers=headers, timeout=30).raise_for_status()
    latencies.append((time.perf_counter() - t0) * 1000)
lat = np.array(latencies)
print(f"\n3. Latency test (30 single-transaction requests, end-to-end from client):")
print(f"   Median {np.median(lat):.0f} ms | 95th percentile {np.percentile(lat, 95):.0f} ms | "
      f"min {lat.min():.0f} ms | max {lat.max():.0f} ms")

# 4. Security test: request without the key
unauth = requests.post(scoring_uri, data=single, headers={"Content-Type": "application/json"}, timeout=30)
print(f"\n4. Security test: request without key -> HTTP {unauth.status_code} "
      f"({'refused, as expected' if unauth.status_code in (401, 403) else 'NOT refused - check!'})")

# Save results for the report
test_summary = {"endpoint": ENDPOINT_NAME, "instance_type": "Standard_F4s_v2",
                "deployment_minutes": 18.7, "consistency_max_diff": float(consistency_diff),
                "latency_ms_median": float(np.median(lat)), "latency_ms_p95": float(np.percentile(lat, 95)),
                "unauthenticated_status": unauth.status_code}
with open(os.path.join("models", "endpoint_test_results.json"), "w") as f:
    json.dump(test_summary, f, indent=2)
print("\nSaved: models/endpoint_test_results.json")

1. Functional test: sample request scored by the cloud endpoint


,true_class,expected_probability,cloud_probability,cloud_flagged
0,1,0.001333,0.001333,False
1,1,0.997782,0.997782,True
2,1,0.972561,0.972561,True
3,0,0.000056,0.000056,False
4,0,0.000135,0.000135,False
5,0,0.000043,0.000043,False



2. Consistency test: 1000 transactions scored in the cloud (1 frauds)
   Maximum difference between cloud and local predictions: 0.0e+00

3. Latency test (30 single-transaction requests, end-to-end from client):
   Median 767 ms | 95th percentile 1146 ms | min 628 ms | max 1187 ms

4. Security test: request without key -> HTTP 403 (refused, as expected)

Saved: models/endpoint_test_results.json


### 8. Monitoring and Logging

This step reviews the operational data the endpoint has collected, covering the monitoring and logging stage of the AI lifecycle:

- **Container logs** show the model loading at start-up and the server-side processing time of each request, as written by the scoring script. Comparing server-side time with the client-side latency measured in the previous step separates model computation from network delay.
- **Endpoint metrics** (request rate, latency, errors, CPU and memory use) are collected automatically by Azure Monitor and Application Insights, and can trigger alerts.

In production, these signals would feed a dashboard and alerting rules (for example, an alert if latency exceeds 300 ms or the error rate rises), and the logged prediction distribution would be monitored for **data drift**, the signal that fraud patterns have changed and the model needs retraining.

In [14]:
import re

# Container logs from the deployment
logs = ml_client.online_deployments.get_logs(name="blue", endpoint_name=ENDPOINT_NAME, lines=300)

startup = [l for l in logs.splitlines() if "Model loaded" in l]
scored = [l for l in logs.splitlines() if "Scored" in l and "transactions" in l]

print("Model start-up log:")
print("  " + (startup[-1].strip() if startup else "(start-up line rotated out of the log window)"))

print(f"\nScoring log lines found: {len(scored)} (showing the last 5)")
for line in scored[-5:]:
    print("  " + line.strip()[-90:])

# Server-side processing time for single-transaction requests
server_ms = [float(m.group(2)) for l in scored
             if (m := re.search(r"Scored (\d+) transactions, \d+ flagged, ([\d.]+) ms", l))
             and m.group(1) == "1"]
if server_ms:
    server_median = float(np.median(server_ms))
    print(f"\nServer-side processing time (single transaction): median {server_median:.1f} ms")
    print(f"Client-side end-to-end latency:                   median {test_summary['latency_ms_median']:.0f} ms")
    print(f"-> Network and HTTPS overhead account for about "
          f"{(1 - server_median / test_summary['latency_ms_median']) * 100:.0f}% of the response time.")
    test_summary["server_ms_median"] = server_median
    with open(os.path.join("models", "endpoint_test_results.json"), "w") as f:
        json.dump(test_summary, f, indent=2)
else:
    print("\nNo per-request timing lines found; showing the last 15 log lines instead:")
    print("\n".join(logs.splitlines()[-15:]))

# Confirm no transaction data appears in the logs (data minimisation check)
leak = any(re.search(r"\bV14\b|\"Amount\"", l) for l in logs.splitlines())
print(f"\nTransaction data found in logs: {leak}  (expected: False)")

Model start-up log:
  (start-up line rotated out of the log window)

Scoring log lines found: 42 (showing the last 5)
  INFO:fraud-endpoint:Scored 1 transactions, 0 flagged, 5.74 ms
  INFO:fraud-endpoint:Scored 1 transactions, 0 flagged, 6.31 ms
  INFO:fraud-endpoint:Scored 1 transactions, 0 flagged, 5.70 ms
  INFO:fraud-endpoint:Scored 1 transactions, 0 flagged, 5.96 ms
  INFO:fraud-endpoint:Scored 1 transactions, 0 flagged, 5.95 ms

Server-side processing time (single transaction): median 6.0 ms
Client-side end-to-end latency:                   median 767 ms
-> Network and HTTPS overhead account for about 99% of the response time.

Transaction data found in logs: False  (expected: False)


### 9. Cost Control: Deleting the Endpoint

Cloud resources are billed for as long as they run, whether or not they receive traffic. The managed endpoint uses a dedicated `Standard_F4s_v2` virtual machine that costs money every hour, so it is deleted now that testing and evidence collection are complete. This illustrates the pay-as-you-go cost model of cloud computing and the operational discipline it requires.

The **data asset, registered model and environment are kept**. They cost almost nothing to store, and they allow the endpoint to be redeployed at any time by re-running the deployment step. In production, cost would be controlled with **autoscaling** (scaling instances with demand, down to a minimum), while a batch endpoint would suit non-urgent, overnight scoring.

In [15]:
deployed_minutes = (time.time() - start) / 60   # from the start of the deployment in Cell 12

print(f"Deleting endpoint {ENDPOINT_NAME} and its deployment...")
ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
print("Endpoint deleted.\n")

remaining = [ep.name for ep in ml_client.online_endpoints.list()]
print(f"Online endpoints still running: {remaining if remaining else 'none'}")

# Assets kept for reproducibility and redeployment
print(f"\nKept in the workspace (minimal storage cost):")
print(f"  Data asset:  {DATA_NAME} v{DATA_VERSION}")
print(f"  Model:       {registered_model.name} v{registered_model.version}")
print(f"  Environment: fraud-xgb-env")

print(f"\nApproximate endpoint lifetime: {deployed_minutes:.0f} minutes "
      f"(about US${deployed_minutes / 60 * 0.20:.2f} at ~US$0.20 per hour for Standard_F4s_v2).")

Deleting endpoint fraud-endpoint-82295 and its deployment...
.............................................................................................Endpoint deleted.

Online endpoints still running: none

Kept in the workspace (minimal storage cost):
  Data asset:  creditcard-fraud-raw v1
  Model:       fraud-xgb-sqrt-weight v1
  Environment: fraud-xgb-env

Approximate endpoint lifetime: 46 minutes (about US$0.15 at ~US$0.20 per hour for Standard_F4s_v2).
